In [ ]:
# Install the required packages:
# - langchain: core LangChain framework including agent middleware support
# - langchain-openai: OpenAI model integration
!pip install -q langchain langchain-openai


In [ ]:
from google.colab import userdata

# AgentState: a typed dict holding the current conversation messages — passed to every middleware
# create_agent: factory function that builds a LangChain agent
from langchain.agents import AgentState, create_agent

# Middleware decorators / classes:
# PIIMiddleware: automatically redacts or masks Personally Identifiable Information (PII) like emails
# ModelRequest / ModelResponse: typed wrappers around what is sent to / received from the LLM
# ToolCallRequest: typed wrapper around a single tool invocation request
# Lifecycle hooks (decorator-based):
#   @before_agent  — runs once before the agent starts processing
#   @after_agent   — runs once after the agent finishes
#   @before_model  — runs before every LLM call
#   @after_model   — runs after every LLM call
#   @wrap_model_call — intercepts the full LLM call (can modify request/response)
#   @wrap_tool_call  — intercepts every tool execution (can modify request/response)
from langchain.agents.middleware import PIIMiddleware, ModelRequest, ModelResponse, ToolCallRequest
from langchain.agents.middleware import after_agent, after_model, before_agent, before_model, wrap_model_call, wrap_tool_call
from langchain.messages import HumanMessage, ToolMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_openai import ChatOpenAI
# Runtime: provides additional context about the running agent (e.g., config, thread id)
from langgraph.runtime import Runtime
from pydantic import SecretStr
from typing import Callable, List

openai_api_key = SecretStr(userdata.get('OPENAI_API_KEY'))

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()


In [ ]:
# A simple tool the agent can call — returns a static interesting fact
@tool
def get_interesting_fact() -> str:
    """
    This tool will discover an interesting fact to you.
    """
    return "The Earth is actually not a perfect sphere."

# NOTE: Middlewares can be implemented by separate functions or by a single class inheriting from `AgentMiddleware`.

# Lifecycle middleware — fires once when the agent starts (before any LLM calls)
@before_agent
def before_agent_func(state: AgentState, runtime: Runtime) -> None:
    print("Event: before_agent")
    print(state)     # current conversation messages
    print(runtime)   # agent runtime context (config, etc.)

# Fires before each individual LLM call
@before_model
def before_model_func(state: AgentState, runtime: Runtime) -> None:
    print("Event: before_model")
    print(state)
    print(runtime)

# Fires after each individual LLM call
@after_model
def after_model_func(state: AgentState, runtime: Runtime) -> None:
    print("Event: after_model")
    print(state)
    print(runtime)

# Fires once when the agent has finished all its work
@after_agent
def after_agent_func(state: AgentState, runtime: Runtime) -> None:
    print("Event: after_agent")
    print(state)
    print(runtime)

# wrap_model_call gives full control over each LLM call.
# `handler` is the next function in the chain — you must call it to actually invoke the model.
@wrap_model_call
def handle_model_call(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]):
    print("Event: model_call")
    print(request)

    # Only force tool on the first LLM step
    # (i.e., when the conversation only contains the original user message)
    if len(request.messages) == 1:
        print("Forcing a specific tool call")
        # override() returns a modified copy of the request with a forced tool choice
        request = request.override(tool_choice=get_interesting_fact.name)

    response = handler(request)

    print("Obtained model response:")
    print(response)

    return response

# wrap_tool_call intercepts every tool execution.
# `handler` is the function that actually runs the tool — you must call it to get the result.
@wrap_tool_call
def handle_tool_call(request: ToolCallRequest, handler: Callable[[ToolCallRequest], ToolMessage]):
    print("Event: tool_call")
    print(request)

    response = handler(request)

    print("Obtained tool message:")
    print(response)

    return response


In [ ]:
# Create the agent and register all middlewares.
# Middlewares are applied in order — each one wraps the next like layers of an onion.
# PIIMiddleware is placed first so PII is redacted before any other processing occurs.
agent = create_agent(
    model=ChatOpenAI(model="gpt-5-nano", api_key=openai_api_key),
    tools=[get_interesting_fact],
    middleware=[
        # Automatically redacts email addresses from messages before they reach the model
        PIIMiddleware(pii_type="email", strategy="redact"),
        before_agent_func,   # Logs when the agent starts
        before_model_func,   # Logs before each LLM call
        after_agent_func,    # Logs when the agent finishes
        after_model_func,    # Logs after each LLM call
        handle_model_call,   # Intercepts + optionally overrides each LLM call
        handle_tool_call     # Intercepts + logs each tool execution
    ]
)


In [ ]:
# Invoke the agent with a message that contains a real email address.
# The PIIMiddleware will redact "maria.popova@example.com" before it reaches the LLM,
# replacing it with a placeholder like "[EMAIL]" to protect sensitive information.
reserve_ticket = agent.invoke(
    input={
        "messages": [HumanMessage("Book two seats under maria.popova@example.com for the play at the theater this evening.")]
    }
)


In [ ]:
# Print the full conversation to see all middleware events, redacted PII, and the agent's final answer
print_conversation(reserve_ticket["messages"])
